# Madrid bus electrification: line, depot and city model
**IE Sustainability Datathon (Kearney) · Deliverable 1 · Main notebook**

This notebook produces the four mandatory outputs `File_1.csv`–`File_4.csv`, the BI visualisation and every check the jury needs. It is built in dependency order: baseline → lines (File 2) → depots (File 3) → prioritisation → scenarios (File 4) → benchmark (File 1) → visualisation → compliance.

**How to read it.** Each section follows the same pattern: a text cell (what, why, sources, V/D/P/A label), a short code cell, then a visible output — a table, a chart or a `CHECK … PASS` line. The jury reads saved outputs only; a step that prints nothing counts as not done.

| Section | Output | Owner |
|---|---|---|
| 0 Setup and parameters | parameter table | [owner] |
| 1 Data acquisition | data register | [owner] |
| 2 Reference day | chosen dates | [owner] |
| 3 Fleet baseline 2025 | baseline, S0 pace | [owner] |
| 4 Line model | File 2 metrics | [owner] |
| 5 Depot allocation | allocation + reconciliation | [owner] |
| 6 Depot power and readiness | File 3 | [owner] |
| 7 Prioritisation | completes File 2 | [owner] |
| 8 Scenarios | File 4 | [owner] |
| 9 Benchmark | File 1 | [owner] |
| 10 Visualisation | HTML + static views | [owner] |
| 11 Assumptions register | register | [owner] |
| 12 Final compliance | read-back + checks | [owner] |

## 0 · Setup and fixed parameters
Libraries, folders and the brief's fixed parameters (section 4.5). Every parameter lives in `P`; no other cell hard-codes a number. Provenance labels: **V** verified · **D** derived · **P** proxy · **A** assumption.

In [ ]:
# Environment: runs as-is in Google Colab (Python 3.10+).
!pip -q install pandas numpy pyproj shapely folium plotly kaleido requests

import os, io, json, zipfile, datetime as dt, hashlib, textwrap
import numpy as np, pandas as pd, requests

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

BASE = "/content/datathon" if os.path.exists("/content") else os.path.abspath("..")
RAW, DERIVED, OUT = (os.path.join(BASE, d) for d in ("data_raw", "data_derived", "outputs"))
for d in (RAW, DERIVED, OUT):
    os.makedirs(d, exist_ok=True)

RUN_DATE = dt.date.today().isoformat()
import sys, platform
print(f"Run date {RUN_DATE} | Python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__}")
print("Folders:", RAW, DERIVED, OUT, sep="\n  ")

In [ ]:
# ---- Data register: every downloaded or manually obtained source is logged here.
DATA_REGISTER = []

def fetch(url, filename, note="", manual=False):
    """Download a public file into data_raw/ and log source, date, size and checksum.
    manual=True: the file was obtained by hand (registration/API); it must already be in data_raw/."""
    path = os.path.join(RAW, filename)
    if not manual and not os.path.exists(path):
        r = requests.get(url, timeout=120)
        r.raise_for_status()
        with open(path, "wb") as f:
            f.write(r.content)
    size = os.path.getsize(path) if os.path.exists(path) else None
    sha = hashlib.sha256(open(path, "rb").read()).hexdigest()[:12] if size else None
    DATA_REGISTER.append(dict(file=filename, url=url, downloaded=RUN_DATE, bytes=size,
                              sha256_12=sha, obtained="manual" if manual else "script", note=note))
    return path

def show_register():
    display(pd.DataFrame(DATA_REGISTER))

# ---- Provenance labels (brief section 4.6): V verified, D derived, P proxy, A assumption.
ASSUMPTIONS = []

def assume(variable, value, label, evidence, impact="", sensitivity=""):
    assert label in {"V", "D", "P", "A"}, "label must be V, D, P or A"
    ASSUMPTIONS.append(dict(variable=variable, value=value, label=label, evidence=evidence,
                            impact=impact, sensitivity=sensitivity))
    return value

# ---- Visible checks: every step prints CHECK ... PASS / FAIL.
CHECKS = []

def check(name, ok, detail=""):
    CHECKS.append(dict(check=name, result="PASS" if ok else "FAIL", detail=detail))
    print(f"CHECK {'PASS' if ok else 'FAIL'} | {name}" + (f" | {detail}" if detail else ""))
    return ok

In [ ]:
# Fixed parameters from the official brief, section 4.5. Do not hard-code these numbers anywhere else.
P = dict(
    kwh_km      = {"Standard": 1.30, "Articulated": 1.85},   # battery energy incl. HVAC
    range_km    = {"Standard": 300,  "Articulated": 240},    # effective daily range (75% of tender minimum)
    cng_kg_km   = {"Standard": 0.45, "Articulated": 0.65},
    charger_kw  = 100,
    window_h    = 7,                                          # 23:00-06:00
    charge_eff  = 0.90,
    bus_price_eur = {"Standard": 615_000, "Articulated": 800_000},  # excl. VAT
    benchmark_kwh_km = 1.30,
    service_days = 365,
    # TODO(owner: emissions) confirm value, unit and boundary against the cited page before submission
    cng_kgco2_per_kg = assume("CO2 factor CNG", 2.7175, "V", "MITECO emission factors (inventory annex VII)", "Files 2, 4"),
    grid_gco2_kwh    = assume("CO2 factor electricity Spain 2025", 108.4, "V", "Red Electrica 2025 generation emissions", "Files 1, 2, 4"),
)
display(pd.DataFrame([(k, v) for k, v in P.items()], columns=["parameter", "value"]))

In [ ]:
# ---- Generic GTFS helpers (shared logic with the benchmark notebook).
def read_gtfs(zip_path, tables=("agency","routes","trips","stop_times","calendar","calendar_dates","shapes","feed_info")):
    feed = {}
    with zipfile.ZipFile(zip_path) as z:
        names = set(z.namelist())
        for t in tables:
            if f"{t}.txt" in names:
                feed[t] = pd.read_csv(z.open(f"{t}.txt"), dtype=str, low_memory=False)
    return feed

def active_service_ids(feed, date_str):
    """Service ids running on date_str (YYYY-MM-DD), applying calendar then calendar_dates exceptions."""
    d = pd.Timestamp(date_str); ymd = d.strftime("%Y%m%d"); dow = d.day_name().lower()
    active = set()
    if "calendar" in feed:
        c = feed["calendar"]
        m = (c["start_date"] <= ymd) & (c["end_date"] >= ymd) & (c[dow] == "1")
        active |= set(c.loc[m, "service_id"])
    if "calendar_dates" in feed:
        cd = feed["calendar_dates"]; cd = cd[cd["date"] == ymd]
        active |= set(cd.loc[cd["exception_type"] == "1", "service_id"])
        active -= set(cd.loc[cd["exception_type"] == "2", "service_id"])
    return active

def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0088
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi, dl = p2 - p1, np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2*r*np.arcsin(np.sqrt(a))

def shape_lengths_km(feed):
    s = feed["shapes"].copy()
    for c in ("shape_pt_lat","shape_pt_lon"): s[c] = s[c].astype(float)
    s["shape_pt_sequence"] = s["shape_pt_sequence"].astype(int)
    s = s.sort_values(["shape_id","shape_pt_sequence"])
    g = s.groupby("shape_id")
    s["seg"] = haversine_km(g["shape_pt_lat"].shift(), g["shape_pt_lon"].shift(), s["shape_pt_lat"], s["shape_pt_lon"])
    return s.groupby("shape_id")["seg"].sum()

def trip_km(feed, trip_ids, dist_unit_m=None):
    """Km per trip. Preferred: shape_dist_traveled (unit must be confirmed: metres or km);
    fallback: geodesic length of the trip's shape. Prints which method each trip used."""
    st = feed["stop_times"][feed["stop_times"]["trip_id"].isin(trip_ids)]
    km = pd.Series(dtype=float)
    if "shape_dist_traveled" in st and st["shape_dist_traveled"].notna().any():
        d = st.assign(x=pd.to_numeric(st["shape_dist_traveled"], errors="coerce")).groupby("trip_id")["x"].agg(lambda v: v.max() - v.min())
        assert dist_unit_m is not None, "Confirm shape_dist_traveled unit (metres or km) and pass dist_unit_m=True/False"
        km = d / 1000 if dist_unit_m else d
    missing = set(trip_ids) - set(km.dropna().index)
    if missing and "shapes" in feed:
        lens = shape_lengths_km(feed)
        t = feed["trips"].set_index("trip_id").loc[list(missing), "shape_id"]
        km = pd.concat([km.dropna(), t.map(lens)])
    print(f"trip_km: {len(km)} trips, {len(missing)} via shape geometry fallback")
    return km

## 1 · Data acquisition
Download every source from its public landing page and log download date, size and checksum. Files that need registration or an API key are obtained manually, placed in `data_raw/` and logged with `manual=True` (brief 5.1). No API keys in this notebook.

In [ ]:
# Fill in the direct file URLs found on each landing page (landing pages are linked in the brief, section 4.1).
SOURCES = {
    # filename: (url, note)
    "emt_gtfs.zip":            ("TODO", "EMT GTFS — https://datos.madrid.es/dataset/900030-0-emt-autobus-gtfs"),
    "emt_supply.csv":          ("TODO", "Buses per day, line and time band — https://datos.madrid.es/dataset/300596-0-emt-lineas-autobuses-tramos"),
    "emt_fleet_history.csv":   ("TODO", "Fleet history — https://datos.madrid.es/dataset/300241-0-emt-flota-autobuses"),
    "emt_calendar.csv":        ("TODO", "Operating calendar — https://datos.madrid.es/dataset/900022-0-calendario-emt-autobus"),
    "emt_ridership.csv":       ("TODO", "Optional: ridership — https://datos.madrid.es/dataset/900028-0-emt-demanda-autobus"),
}
for fn, (url, note) in SOURCES.items():
    if url != "TODO":
        fetch(url, fn, note)
show_register()

## 2 · Reference day
Rule (brief section 2): a Tuesday or Wednesday, regular working day in EMT's calendar, outside holidays and the summer timetable, no strikes or events, inside GTFS validity.

**Known gap:** the GTFS feed is valid from 6 Nov 2026 while the supply data ends Sep 2026. The brief allows `peak_buses` from the closest equivalent weekday when the reference date is not yet published; document both dates here and in the report.

In [ ]:
# TODO(owner: reference day) inspect feed_info / calendar validity and the calendar dataset, then set:
REF_DATE = "YYYY-MM-DD"          # GTFS reference day (Madrid File 1 and File 2)
SUPPLY_DATE = "YYYY-MM-DD"       # closest equivalent weekday in the supply data, if different
assume("Reference day", REF_DATE, "A", "Tue/Wed, regular day, within GTFS validity")
assume("Supply-data day for peak_buses", SUPPLY_DATE, "A", "Closest equivalent weekday published (brief File 2 rule)")
print("Reference day:", REF_DATE, "| supply day:", SUPPLY_DATE)

## 3 · Fleet baseline at 31 Dec 2025 and 2021–25 pace
Rebuild the fleet at 31 Dec 2020 and 31 Dec 2025 from the fleet history (active and retirement dates), by technology and size. Reconcile against the 2025 accounts note and EMT's *Principales cifras*; state which source is used and why. Decide how plug-in CNG hybrids are counted so `n_bev + n_h2 + n_cng = fleet_total` holds. Print the CNG age profile (it drives retirements in File 4).

In [ ]:
# TODO(owner: baseline)
# decode fleet-file codes (technology, length), build fleet at 2020-12-31 and 2025-12-31, reconciliation table, age chart


In [ ]:
BASELINE_2025 = dict(fleet_total=None, n_bev=None, n_h2=None, n_cng=None, std_share=None)   # TODO fill from the cell above
ZE_2020 = None                                                                                  # TODO
S0_PACE_UNADJUSTED = None  # = (ZE 2025 - ZE 2020) / 5
print("Baseline 31 Dec 2025:", BASELINE_2025, "| unadjusted S0 pace:", S0_PACE_UNADJUSTED)

## 4 · Line model (File 2 metrics)
For every EMT line with ≥1 trip on the reference day: scheduled in-service km (shape distances, cross-checked against KML), `peak_buses` = max buses across the supply data's time bands (mandatory source), bus-type proxy, energy per bus-day, charging screen (Rule 2) and `co2_avoided_t_year`. Lines in GTFS without supply records stay in File 2 with a documented proxy.

**Per-bus sensitivity (Rule 2):** report which lines exceed range under an alternative block proxy (first-to-last-trip span, or off-peak buses).

In [ ]:
# TODO(owner: line model)
# load GTFS, select active trips on REF_DATE, compute daily_km per line (confirm shape_dist_traveled unit), map supply line ids to GTFS route ids


In [ ]:
# TODO(owner: line model)
# peak_buses from supply data on SUPPLY_DATE; list lines with no supply record and their proxy


In [ ]:
# TODO(owner: line model)
# bus_type proxy (e.g. articulated share from fleet history assigned to lines with highest riders per peak bus); log with assume(..., 'P', ...)


In [ ]:
def line_metrics(lines):
    """lines: DataFrame with line_id, daily_km, peak_buses, bus_type. Applies the brief's fixed formulas."""
    df = lines.copy()
    df["km_per_bus_day"] = df.daily_km / df.peak_buses
    df["energy_kwh_bus_day"] = df.km_per_bus_day * df.bus_type.map(P["kwh_km"])
    within = df.km_per_bus_day <= df.bus_type.map(P["range_km"])
    df["charging_strategy"] = np.where(within, "Depot", "Opportunity")  # TODO justify Opportunity vs Not_feasible above range
    cng_kg_co2_km = df.bus_type.map(P["cng_kg_km"]) * P["cng_kgco2_per_kg"]
    ele_kg_co2_km = df.bus_type.map(P["kwh_km"]) / P["charge_eff"] * P["grid_gco2_kwh"] / 1000
    df["co2_avoided_t_year"] = df.daily_km * P["service_days"] * (cng_kg_co2_km - ele_kg_co2_km) / 1000
    return df
# lines = line_metrics(lines); display(lines.describe()); print(lines.charging_strategy.value_counts())

In [ ]:
# TODO(owner: line model)
# per-bus sensitivity: first-to-last-trip span or off-peak buses -> lines exceeding range


## 5 · Depot allocation and fleet reconciliation
Assign each line to one primary depot with a transparent rule (e.g. terminal-to-depot distance with a capacity check), compare with an alternative rule and with the 2021 per-depot counts. Reconcile active vs physical fleet with a reserve ratio: physical ≈ active ÷ (1 − r). Never add reserve buses into `buses_assigned`. **Open item:** La Elipa's operating status (closed Mar 2022 for rebuilding).

In [ ]:
# TODO(owner: depots)
# geocode the five depots (addresses from the 2025 recruitment syllabus), label coordinates V or P


In [ ]:
# TODO(owner: depots)
# primary rule + alternative rule, comparison with 2021 counts, reserve-ratio reconciliation table


## 6 · Depot energy, power and readiness (File 3)
`night_energy_mwh = Σ peak_buses × energy_kwh_bus_day / 1000` (all assigned lines); `peak_mw_unmanaged = buses × 100 kW`; `peak_mw_managed = energy ÷ (7 h × 0.90)`. Connection: Sanchinarro 8 MW main + 5 MW backup is **Evidenced**; other depots use a documented proxy (**Estimated**). Readiness per Rule 4, plus at least one alternative connection hypothesis.

In [ ]:
def depot_table(file2, depots):
    """depots: depot_name, latitude, longitude, connection_mw_estimate, connection_evidence."""
    g = file2.assign(e=file2.peak_buses * file2.energy_kwh_bus_day / 1000).groupby("depot_name").agg(
        n_lines_assigned=("line_id", "count"), buses_assigned=("peak_buses", "sum"),
        night_energy_mwh=("e", "sum"), conversion_year=("electrification_year", "max")).reset_index()
    d = depots.merge(g, on="depot_name", how="left")
    d["peak_mw_unmanaged"] = d.buses_assigned * P["charger_kw"] / 1000
    d["peak_mw_managed"] = d.night_energy_mwh / (P["window_h"] * P["charge_eff"])
    c = d.connection_mw_estimate
    d["readiness_status"] = np.where(d.peak_mw_managed <= c, "Ready", np.where(d.peak_mw_managed <= 1.5 * c, "Upgrade", "Critical"))
    return d

In [ ]:
# TODO(owner: depots)
# connection proxies for the other depots (UFD / e-distribución / i-DE node data, published chargers x simultaneity); alternative-connection sensitivity table


## 7 · Prioritisation (completes File 2)
Multi-criteria `priority_rank` (e.g. charging feasibility, depot readiness, CO₂ avoided, exposure or ridership), then `electrification_year` under S1 so a better rank never has a later year. Use 2025 only for lines already fully zero-emission at the baseline (45 electric lines + line 145 hydrogen, per evidence).

In [ ]:
# TODO(owner: prioritisation)
# weights and scoring table, priority_rank, electrification_year; write File 2 and File 3


In [ ]:
# file2.to_csv(os.path.join(OUT, "File_2.csv"), index=False, encoding="utf-8")
# file3.to_csv(os.path.join(OUT, "File_3.csv"), index=False, encoding="utf-8")

## 8 · Scenarios 2026–2035 (File 4)
**S0** – 2021–25 pace, adjusted for tenders and deliveries (report both paces). **S1** – 100 % zero-emission by 31 Dec 2030, held to 2035. Explicit yearly purchases and retirements (not inferred from fleet change); service-km allocation by technology; standard/articulated km mix; grid-input energy (÷ 0.90); CNG tonnes; CO₂; avoided CO₂ vs counterfactual (2025 ZE share frozen); capex in constant 2025 €, excl. VAT. Then the earliest credible year and binding constraint, with ≥2 sensitivities.

In [ ]:
# TODO(owner: scenarios)
# purchase/retirement schedules per scenario; fleet mix; energy, fuel, CO2, counterfactual, capex; write File 4


In [ ]:
# TODO(owner: scenarios)
# earliest credible year: pace and budget vs depot/grid vs deliveries; sensitivities (e.g. grid decarbonisation, connection proxy, bus life)


## 9 · European benchmark (File 1)
Loads `data_derived/benchmark_gtfs_intermediate.csv` from notebook 02 (Helsinki fully reproducible; Berlin, Paris, Brussels). Adds Madrid's row from File 2 and the 2025 baseline, plus cited fleet, zero-emission target, charging model and national grid factor per city. Madrid target: 2033 (City of Madrid, Aug 2023).

In [ ]:
# TODO(owner: benchmark)
# merge intermediate + cited fields; energy_full_bev_mwh_day = daily_bus_km x 1.30 / 1000; write File 1


## 10 · BI visualisation
One self-contained HTML (data embedded, opens by double-click): (i) route map by charging strategy, (ii) depot map with readiness colours and Evidenced/Estimated flag, (iii) benchmark, (iv) S0 vs S1. Each view is also displayed here as a static image (brief 5.3).

In [ ]:
# TODO(owner: visualisation)
# build outputs/madrid_bus_electrification.html from Files 1-4 only; display each view inline; export PNGs


## 11 · Assumptions register
Every value logged with `assume()` across the notebook, with its label, evidence, impact and sensitivity.

In [ ]:
display(pd.DataFrame(ASSUMPTIONS))

## 12 · Final compliance cell
Reads `File_1.csv`–`File_4.csv` back **as saved** and prints columns, data types, row counts and `head(10)` for each, plus the Rule 3 cross-file checks and the File 4 identities. Mandatory (brief 5.1); a missing printout risks disqualification.

In [ ]:
# FINAL COMPLIANCE CELL (brief 5.1 and Rule 3). Reads the four CSVs back AS SAVED and prints everything.
F_COLS = {
 "File_1.csv": ["city","operator","reference_date","n_bus_lines","daily_bus_km","fleet_total","ze_fleet","ze_share_pct",
                "fleet_data_year","ze_target_year","charging_model","energy_full_bev_mwh_day","grid_co2_g_kwh","source_urls"],
 "File_2.csv": ["line_id","daily_km","peak_buses","km_per_bus_day","bus_type","energy_kwh_bus_day","charging_strategy",
                "depot_name","priority_rank","electrification_year","co2_avoided_t_year"],
 "File_3.csv": ["depot_name","latitude","longitude","n_lines_assigned","buses_assigned","night_energy_mwh","peak_mw_unmanaged",
                "peak_mw_managed","connection_mw_estimate","connection_evidence","readiness_status","conversion_year"],
 "File_4.csv": ["year","scenario","fleet_total","n_bev","n_h2","n_cng","ze_share_pct","electricity_mwh","cng_t","co2_t",
                "co2_avoided_t","capex_meur"],
}
CHECKS.clear()
F = {}
for name, cols in F_COLS.items():
    path = os.path.join(OUT, name)
    print("=" * 100, f"\n{name}")
    if not check(f"{name} exists", os.path.exists(path)):
        continue
    df = pd.read_csv(path, encoding="utf-8", dtype={"line_id": str})
    F[name] = df
    print(f"rows: {len(df)} | columns: {len(df.columns)}")
    print(df.dtypes.to_string())
    display(df.head(10))
    check(f"{name} has exactly the required columns, in order", list(df.columns) == cols,
          "" if list(df.columns) == cols else f"got {list(df.columns)}")

close = lambda a, b, tol=1e-3: abs(float(a) - float(b)) <= tol * max(1.0, abs(float(b)))
print("=" * 100, "\nCross-file and internal checks")
if {"File_2.csv", "File_3.csv"} <= F.keys():
    f2, f3 = F["File_2.csv"], F["File_3.csv"]
    check("File 2: priority_rank unique 1..N", sorted(f2.priority_rank) == list(range(1, len(f2) + 1)))
    s = f2.sort_values("priority_rank")
    check("File 2: better rank never has a later electrification_year", s.electrification_year.is_monotonic_increasing)
    check("File 2: electrification_year within 2025-2030", f2.electrification_year.between(2025, 2030).all())
    rng = f2.bus_type.map(P["range_km"])
    depot_ok = ((f2.km_per_bus_day <= rng) == (f2.charging_strategy == "Depot")).all()
    check("File 2: Depot if and only if km_per_bus_day <= range (Rule 2)", depot_ok)
    check("File 2: energy = km_per_bus_day x kWh/km", np.allclose(f2.energy_kwh_bus_day, f2.km_per_bus_day * f2.bus_type.map(P["kwh_km"]), rtol=1e-3))
    check("All File 2 depot names appear in File 3", set(f2.depot_name) <= set(f3.depot_name))
    agg = f2.assign(e=f2.peak_buses * f2.energy_kwh_bus_day / 1000).groupby("depot_name").agg(
        n=("line_id", "count"), b=("peak_buses", "sum"), e=("e", "sum"), y=("electrification_year", "max"))
    m = f3.set_index("depot_name").join(agg, how="left")
    m = m[m.n.notna()]
    check("File 3 n_lines_assigned reproducible from File 2", (m.n_lines_assigned == m.n).all())
    check("File 3 buses_assigned = sum of File 2 peak_buses", (m.buses_assigned == m.b).all())
    check("File 3 night_energy_mwh reproducible from File 2", np.allclose(m.night_energy_mwh, m.e, rtol=1e-3))
    check("File 3 conversion_year = max electrification_year", (m.conversion_year == m.y).all())
    check("File 3 peak_mw_unmanaged = buses x 100 kW", np.allclose(f3.peak_mw_unmanaged, f3.buses_assigned * P["charger_kw"] / 1000, rtol=1e-3))
    check("File 3 peak_mw_managed = energy / (7 x 0.90)", np.allclose(f3.peak_mw_managed, f3.night_energy_mwh / (P["window_h"] * P["charge_eff"]), rtol=1e-3))
    rd = np.where(f3.peak_mw_managed <= f3.connection_mw_estimate, "Ready",
                  np.where(f3.peak_mw_managed <= 1.5 * f3.connection_mw_estimate, "Upgrade", "Critical"))
    check("File 3 readiness_status follows Rule 4", (rd == f3.readiness_status).all())
    check("File 3 connection_evidence in {Evidenced, Estimated}", f3.connection_evidence.isin(["Evidenced", "Estimated"]).all())
if {"File_1.csv", "File_2.csv"} <= F.keys():
    f1, f2 = F["File_1.csv"], F["File_2.csv"]
    mad = f1[f1.city == "Madrid"]
    if check("File 1 has a Madrid row", len(mad) == 1):
        mad = mad.iloc[0]
        check("File 1 has Madrid + >= 3 eligible capitals", len(f1) >= 4 and set(f1.city) - {"Madrid"} <= {"London","Paris","Berlin","Brussels","Helsinki","Copenhagen"})
        check("Madrid daily_bus_km = sum(File 2 daily_km)", close(mad.daily_bus_km, f2.daily_km.sum()))
        check("Madrid n_bus_lines = count(File 2 rows)", int(mad.n_bus_lines) == len(f2))
        check("Madrid reference_date = File 2 reference date", str(mad.reference_date) == REF_DATE, f"{mad.reference_date} vs {REF_DATE}")
        check("Madrid fleet_total = 2025 baseline", int(mad.fleet_total) == BASELINE_2025["fleet_total"])
        check("Madrid ze_fleet = 2025 baseline", int(mad.ze_fleet) == BASELINE_2025["n_bev"] + BASELINE_2025["n_h2"])
    check("File 1 ze_share_pct = ze_fleet / fleet_total x 100", np.allclose(f1.ze_share_pct, (f1.ze_fleet / f1.fleet_total * 100).round(1), atol=0.051))
    k = f1.daily_bus_km.notna()
    check("File 1 energy_full_bev_mwh_day = daily_bus_km x 1.30 / 1000", np.allclose(f1.energy_full_bev_mwh_day[k], f1.daily_bus_km[k] * P["benchmark_kwh_km"] / 1000, rtol=1e-3))
    check("File 1 source_urls populated for every row", f1.source_urls.fillna("").str.len().gt(0).all())
if "File_4.csv" in F:
    f4 = F["File_4.csv"]
    check("File 4 has exactly 20 rows (2026-2035 x S0, S1)", len(f4) == 20 and set(zip(f4.year, f4.scenario)) == {(y, s) for y in range(2026, 2036) for s in (0, 1)})
    check("File 4: n_bev + n_h2 + n_cng = fleet_total (every row)", (f4.n_bev + f4.n_h2 + f4.n_cng == f4.fleet_total).all())
    check("File 4: ze_share_pct = 100 x (n_bev + n_h2) / fleet_total", np.allclose(f4.ze_share_pct, (100 * (f4.n_bev + f4.n_h2) / f4.fleet_total).round(1), atol=0.051))
    s1 = f4[(f4.scenario == 1) & (f4.year >= 2030)]
    check("File 4: S1 ze_share_pct = 100 from 2030 to 2035", (s1.ze_share_pct == 100).all())
print("\nBaseline 31 Dec 2025 used by Files 1 and 4:", BASELINE_2025 if "BASELINE_2025" in globals() else "NOT SET")
summary = pd.DataFrame(CHECKS)
display(summary)
print(f"\n{(summary.result == 'PASS').sum()} PASS / {(summary.result == 'FAIL').sum()} FAIL")